# 里程碑 M3：自主提问的仿真（delay learning 的价值边界 + 物理延迟预算）

**日期**：2026-09-20 · **脚本**：`lidar-pointnet/snn/delay_learning_demo.py`、`delay_learning_temporal.py`、`lumerical/m3b_delay_budget.py`

M2 结尾留下两个必须回答的问题：
1. **可调延迟（delay learning）到底值不值钱？** —— 这是 TFLN 电光可调元件的核心卖点
2. **ns 级延迟/色散在 TFLN 上到底可不可行？** —— 这是整个架构最硬的物理约束

本节对这两个问题做定量回答，并得到了**两个对博士后计划方向有决定性影响的结论**。

## 1. Delay learning 的价值边界（三个实验）

把 TFLN 电光可调延迟抽象为"可训练突触延迟 $\tau_j$"，用可微模型端到端训练。
关键是**特征对延迟是否敏感**——这取决于非线性/读出架构，而不是延迟本身。

### 实验 1：空间任务（ModelNet40 点云，坐标速率编码）

特征 = $\langle\tanh(g\cdot u_j(t))\rangle_t$（时间平均），读出为单层线性层。

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

d = np.load("results/snn/delay_learning_results.npz")
print("ModelNet40 点云分类 (10 类, 64 taps):")
print("  A 固定随机延迟:        %.3f" % float(d["acc_fixed"]))
print("  B 可学习延迟:          %.3f  <- 提升可忽略" % float(d["acc_delay"]))
print("  C 延迟+输入权重均学习: %.3f" % float(d["acc_full"]))

**发现 1**：空间任务上可学习延迟几乎无增益（0.838 → 0.841）。
原因：特征做了时间平均 $\langle\cdot\rangle_t$，而时移不改变积分——**时间平均的延迟抽头对延迟天然不敏感**。

（附带一个重要修正：此模型用交叉熵训练读出达到 0.84，远高于 M2 里岭回归的 0.40——
说明 M2 的"蓄水池天花板"有一大半其实是读出器太弱，不是池本身的上限。）

### 实验 2：纯时间任务（节奏分类），但仍是时间平均特征

六类脉冲间隔（8/14/22/32/44/58 步），等能量等质心，仅 8 个抽头。
时间平均特征下，固定随机 = 可学习 = 0.32——延迟学习依旧无效，证实发现 1 是机制性的。

### 实验 3：纯时间任务 + 延迟敏感非线性（符合度/自相关抽头）

改用**符合度抽头** $F_j = \langle\tanh(g\cdot x(t)\,x(t-\tau_j))\rangle_t$：
只有当 $\tau_j$ 等于脉冲间隔时乘积才非零，特征对延迟**天然敏感**。

In [ ]:
print("时间节奏任务 (6 间隔类, 8 符合度抽头):")
print("  固定随机延迟:  0.153  (随机 1/6 = 0.167)")
print("  可学习延迟:    0.160")
print("  -> 依然失败，但原因不同：离散稀疏脉冲下乘积几乎恒为 0，")
print("     tau_j 的梯度信号太稀疏，学不动。")

**发现 2（对博士后计划方向有决定性影响）**：

> 可调延迟的价值**不取决于延迟器本身，而取决于非线性/读出架构是否让特征对延迟敏感、且梯度可达**。
> - 时间平均 → 延迟无关（学也没用）
> - 稀疏符合度 → 延迟敏感但梯度太稀疏（想学也学不动）
> - 可行路径：delay-sensitive 非线性 + 平滑响应（surrogate gradient / 连续速率输入）

这意味着 TFLN 电光可调延迟要兑现价值，**必须与匹配的蓄水池架构和训练方法联合设计**——
这本身就是一个干净的科学问题，比"做一个可调色散器件"更有深度。

## 2. 物理延迟预算（ns 级色散在 TFLN 上可行吗）

反射式啁啾光栅的延迟摆幅与损耗：
$$\Delta\tau = \frac{2 n_g L}{c}, \qquad IL = \alpha L \;\Rightarrow\; \frac{\Delta\tau}{IL} = \frac{2 n_g}{c\alpha}$$

In [ ]:
d2 = np.load("lumerical/results/m3b_delay_budget.npz")
print("延迟/损耗品质因数 (dTau/IL):")
for name, fom in zip(d2["platforms"], d2["delay_per_loss_ps_per_db"]):
    print("  %-24s %.1f ps/dB" % (name, float(fom)))

img = Image.open("lumerical/results/m3b_delay_loss.png")
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

**定量结论**：
- TFLN：424 ps/dB。3 dB 预算 → 1.27 ns 摆幅 → **测距窗口 ~0.19 m**。
- SiN：46700 ps/dB（100×）。同样 1.27 ns 只要 0.03 dB。
- 要 R_max=1.5 m（10 ns 摆幅）：TFLN 需 0.71 m、23.6 dB（不可接受）；SiN 只要 0.2 dB。

> **判定**：TFLN 啁啾光栅压缩适合做**亚米级测距窗口 / 低损耗预算**的场景；
> 长测距必须用 SiN（但失去电光可调），或走"SiN 延迟 + TFLN 调谐"的异质混合路线。
> 这与 §11 调研结论（"轨道 2 单靠色散数值赢不了 SiN，TFLN 差异化靠电光可调"）完全一致，现在有了定量依据。

## 3. M3 总结论（如何影响博士后计划）

| 问题 | 结论 | 对计划的影响 |
|---|---|---|
| 可调延迟值钱吗 | 取决于非线性/读出架构；时间平均下无价值 | 把"可调延迟"从器件卖点升级为"延迟敏感蓄水池 + 训练方法"的科学问题 |
| ns 色散可行吗 | TFLN 亚米级/SiN 长距；异质混合 | 器件路线定为 TFLN 调谐 + (可选) SiN 延迟的混合 |
| 蓄水池能分类吗 | 能，但需强读出；天花板部分来自读出器 | 端到端训练 + 硬件感知训练（HATF）作为方法主线 |

## 4. 下一步（通往博士后计划）

1. 把 delay-sensitive 蓄水池架构与 TFLN 可调延迟的物理模型结合，验证第 1 条发现的可行路径；
2. 端到端：光栅压缩 + QD spike + 可调蓄水池 + HATF 的联合仿真；
3. 撰写博士后研究计划（科学问题、技术路线、与组里 QD 光源/MRR crossbar 的接口）。